# This notebook downloads all the trained models and tests their functionalities.

In [47]:
# Change directory to the project root.
import os
os.chdir('../src')

In [48]:
!pwd

'pwd' is not recognized as an internal or external command,
operable program or batch file.


In [49]:
# Download weights and check the availablity.
from ml_manager.utils.downloader import download_all_models, check_model_weights, download_from_google_drive

### Download all models in one ZIP file

In [50]:
success = download_all_models()

ball_detection model exists.
action_detection model exists.
court_detection model exists.
game_state model exists.
INFO: All model weights already exist


In [51]:
_ = check_model_weights()

ball_detection model exists.
action_detection model exists.
court_detection model exists.
game_state model exists.


### Check initialization of the models with the downloaded weights.

In [52]:
# Check if the models are initialized well.
from ml_manager.ml_manager import MLManager

ml_engine = MLManager()

INFO: ML Manager initialized on device: cuda
INFO: Using default weights configuration
INFO: Initializing deep learning models...
ball_detection model exists.
action_detection model exists.
court_detection model exists.
game_state model exists.
INFO: All model weights are available locally
INFO: Initializing BallDetector with model: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\ball\weights\best.pt
INFO: Loading YOLO model from: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\ball\weights\best.pt
INFO: Moving YOLO model to device: cuda
SUCCESS: Ball segmentation model loaded: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\ball\weights\best.pt
INFO: Initializing CourtSegmentation with model: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\court\weights\best.pt
INFO: Loading YOLO model from: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\court\weights\best.pt
INFO: Moving YOLO model to device: cuda
SUCCESS: Court segmentation mod

Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

SUCCESS: Loaded VideoMAE model from c:/Users/hazar/Desktop/CS/volleyball_analytics/src/weights/game_state on device cuda
SUCCESS: Game state classification model loaded: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\game_state
SUCCESS: All models initialized successfully!


### Download sample video from google drive and check the results on the video.

In [53]:
from pathlib import Path
from ml_manager.utils.downloader import download_from_google_drive
from demo import run_object_detection, run_video_classification

gid = '1kmPY_a2OZ9m484Ypx-DDuK_pE0QLxxaR'
filename = 'tokyo2020-poland-vs-iran.mp4'
output_path = 'output'

Path(output_path).mkdir(parents=True, exist_ok=True)
sample_video = Path(output_path).joinpath(filename)

if not sample_video.exists():
    success = download_from_google_drive(file_id=gid, output_path=output_path)

obj_det_output = 'output/pol-vs-irn-obj-det.mp4'
vid_cls_output = 'output/pol-vs-irn-video-classification.mp4'

# Leave these commented out for now. They need extra arguments.
# run_object_detection(...)
# run_video_classification(...)




In [54]:
import cv2
import numpy as np
import supervision as sv
from pathlib import Path

from rich.progress import Progress
from ml_manager.ml_manager import MLManager


def run_video_classification(video_path: str, output_path: str) -> None:
    """Classify game state every 30 frames and write an annotated video."""
    print("Initializing ML Manager...")
    ml_manager = MLManager()

    if not ml_manager.is_model_available('game_state_classification'):
        raise RuntimeError("Game state classification model not available")

    print("Opening video...")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video properties: {width}x{height}, {fps} FPS, {total_frames} frames")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    if not out.isOpened():
        raise RuntimeError(f"Could not create output video: {output_path}")

    classification_interval = 30
    frame_buffer = []
    current_game_state = "Unknown"
    current_confidence = 0.0
    frame_count = 0

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    thickness = 2

    print("Processing video frames...")
    with Progress() as progress:
        task = progress.add_task("[green]Processing video...", total=total_frames)

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            progress.update(task, advance=1)

            # Collect frames, classify once we have a full batch, then reset
            frame_buffer.append(frame.copy())
            if len(frame_buffer) == classification_interval:
                result = ml_manager.classify_game_state(frame_buffer)
                current_game_state = result.predicted_class
                current_confidence = result.confidence
                frame_buffer = []  # <- the missing reset

                progress.update(
                    task,
                    description=(
                        f"[green]Frame {frame_count} | "
                        f"State: {current_game_state} | "
                        f"Conf: {current_confidence:.3f}"
                    ),
                )

            # Annotate EVERY frame with the latest prediction
            annotated_frame = frame.copy()

            state_text = f"Game State: {current_game_state}"
            confidence_text = f"Confidence: {current_confidence:.3f}"
            frame_text = f"Frame: {frame_count}"

            text_size = cv2.getTextSize(state_text, font, font_scale, thickness)[0]

            # Pick a border color from the state name
            s = str(current_game_state).lower().replace("-", " ").replace("_", " ")
            if s == "play":
                color = (0, 255, 0)
            elif s == "no play":
                color = (0, 0, 255)
            elif s == "service":
                color = (255, 0, 255)
            else:
                color = (255, 255, 255)

            cv2.rectangle(annotated_frame, (10, 10), (text_size[0] + 20, 100), (0, 0, 0), -1)
            cv2.rectangle(annotated_frame, (10, 10), (text_size[0] + 20, 100), color, 3)

            cv2.putText(annotated_frame, state_text, (15, 35), font, font_scale, (255, 255, 255), thickness)
            cv2.putText(annotated_frame, confidence_text, (15, 60), font, font_scale, (255, 255, 255), thickness)
            cv2.putText(annotated_frame, frame_text, (15, 85), font, font_scale, (255, 255, 255), thickness)

            out.write(annotated_frame)

    cap.release()
    out.release()
    if hasattr(ml_manager, "cleanup"):
        ml_manager.cleanup()

    print(f"Video classification completed. Frames written: {frame_count}. Output saved to: {output_path}")

In [55]:
gid = '1kmPY_a2OZ9m484Ypx-DDuK_pE0QLxxaR'
filename = 'tokyo2020-poland-vs-iran.mp4'
output_path = 'output'

Path(output_path).mkdir(parents=True, exist_ok=True)
sample_video = Path(output_path).joinpath(filename)

if not sample_video.exists():
    success = download_from_google_drive(file_id=gid, output_path=output_path)


obj_det_output = 'output/pol-vs-irn-obj-det.mp4'
vid_cls_output = 'output/pol-vs-irn-video-classification.mp4'


run_video_classification(video_path=str(sample_video), output_path=vid_cls_output)

Initializing ML Manager...
INFO: ML Manager initialized on device: cuda
INFO: Using default weights configuration
INFO: Initializing deep learning models...
ball_detection model exists.
action_detection model exists.
court_detection model exists.
game_state model exists.
INFO: All model weights are available locally
INFO: Initializing BallDetector with model: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\ball\weights\best.pt
INFO: Loading YOLO model from: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\ball\weights\best.pt
INFO: Moving YOLO model to device: cuda
SUCCESS: Ball segmentation model loaded: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\ball\weights\best.pt
INFO: Initializing CourtSegmentation with model: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\court\weights\best.pt
INFO: Loading YOLO model from: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\court\weights\best.pt
INFO: Moving YOLO model to device: cuda
SUCC

Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

SUCCESS: Loaded VideoMAE model from c:/Users/hazar/Desktop/CS/volleyball_analytics/src/weights/game_state on device cuda
SUCCESS: Game state classification model loaded: c:\Users\hazar\Desktop\CS\volleyball_analytics\src\weights\game_state
SUCCESS: All models initialized successfully!


Output()

Opening video...
Video properties: 1920x1080, 50 FPS, 2351 frames
Processing video frames...


Video classification completed. Frames written: 2101. Output saved to: output/pol-vs-irn-video-classification.mp4


In [56]:
# Testing gamestate classifier.
import numpy as np

frame = np.ones((224, 224, 3), dtype=np.uint8)
frames = [frame.copy() for _ in range(16)]


ml_engine.classify_game_state(frames)

GameStateResult(predicted_class=<GameState.PLAY: 'play'>, confidence=1.0)